In [1]:
import subprocess
import sys

def _pip(*packages):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL,
    )

try:
    import faiss
except ImportError:
    print("[setup] Устанавливаю faiss-cpu…")
    _pip("faiss-cpu")

try:
    import sentence_transformers
except ImportError:
    print("[setup] Устанавливаю sentence-transformers…")
    _pip("sentence-transformers")

import json
import pickle
from collections import Counter
from pathlib import Path

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ── Настройки ─────────────────────────────────────────────────────────────────

INPUT_FILES = [
    "/kaggle/input/datasets/volodimirzov/handyman/smarthandyman_ifixit.jsonl",
    "/kaggle/input/datasets/volodimirzov/handyman/smarthandyman_wikihow.jsonl",
    "/kaggle/input/datasets/volodimirzov/handyman/smarthandyman_bobvila.jsonl",
    "/kaggle/input/datasets/volodimirzov/handyman/smarthandyman_mastergrad.jsonl",
]

RAG_DIR       = Path("/kaggle/working/rag_store")
INDEX_PATH    = RAG_DIR / "index.faiss"
CHUNKS_PATH   = RAG_DIR / "chunks.pkl"
METADATA_PATH = RAG_DIR / "metadata.pkl"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHUNK_SIZE      = 400
OVERLAP         = 80
MIN_CHUNK_WORDS = 30
BATCH_SIZE      = 64

# ──────────────────────────────────────────────────────────────────────────────

RAG_DIR.mkdir(exist_ok=True)

# ── 1. Загрузка документов ────────────────────────────────────────────────────

documents = []
seen_urls = set()

for path in INPUT_FILES:
    f = Path(path)
    if not f.exists():
        print(f"⚠ Файл не найден: {path}")
        continue
    count = 0
    with f.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                doc = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = doc.get("text", "").strip()
            url  = doc.get("url", "")
            if not text or len(text) < 50:
                continue
            if url and url in seen_urls:
                continue
            if url:
                seen_urls.add(url)
            documents.append({
                "text":     text,
                "source":   url or path,
                "title":    doc.get("title", ""),
                "provider": doc.get("source", f.stem),
                "domain":   doc.get("domain", ""),
            })
            count += 1
    print(f"  {f.name}: {count} документов")

print(f"\nВсего документов: {len(documents)}")

# ── 2. Чанкование ─────────────────────────────────────────────────────────────

all_chunks = []
metadata   = []
step = CHUNK_SIZE - OVERLAP

for doc in documents:
    words = doc["text"].split()
    if not words:
        continue
    chunks = []
    for i in range(0, len(words), step):
        chunk_words = words[i: i + CHUNK_SIZE]
        if len(chunk_words) < MIN_CHUNK_WORDS:
            if chunks:
                chunks[-1] += " " + " ".join(chunk_words)
            break
        chunks.append(" ".join(chunk_words))
    for chunk in chunks:
        all_chunks.append(chunk)
        metadata.append({
            "source":   doc["source"],
            "title":    doc["title"],
            "provider": doc["provider"],
            "domain":   doc["domain"],
        })

print(f"Чанков: {len(all_chunks)}")
print("По источникам:")
for p, n in Counter(m["provider"] for m in metadata).most_common():
    print(f"  {p:25s}: {n}")

# ── 3. Эмбеддинги ─────────────────────────────────────────────────────────────

print(f"\nСчитаю эмбеддинги…")
model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = model.encode(
    all_chunks,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

print(f"Эмбеддинги: {embeddings.shape}")

# ── 4. FAISS индекс ───────────────────────────────────────────────────────────

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"Проиндексировано векторов: {index.ntotal}")

# ── 5. Сохранение ─────────────────────────────────────────────────────────────

faiss.write_index(index, str(INDEX_PATH))
with CHUNKS_PATH.open("wb") as f:
    pickle.dump(all_chunks, f)
with METADATA_PATH.open("wb") as f:
    pickle.dump(metadata, f)

print(f"\n✅ Готово:")
print(f"   {INDEX_PATH}")
print(f"   {CHUNKS_PATH}")
print(f"   {METADATA_PATH}")

[setup] Устанавливаю faiss-cpu…
  smarthandyman_ifixit.jsonl: 351 документов
  smarthandyman_wikihow.jsonl: 338 документов
  smarthandyman_bobvila.jsonl: 1051 документов
  smarthandyman_mastergrad.jsonl: 340 документов

Всего документов: 2080
Чанков: 11445
По источникам:
  bobvila                  : 6265
  mastergrad               : 3083
  wikihow                  : 1528
  smarthandyman_ifixit     : 569

Считаю эмбеддинги…


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/179 [00:00<?, ?it/s]

Эмбеддинги: (11445, 384)
Проиндексировано векторов: 11445

✅ Готово:
   /kaggle/working/rag_store/index.faiss
   /kaggle/working/rag_store/chunks.pkl
   /kaggle/working/rag_store/metadata.pkl
